# Adaptive Nervous System v0.2.0 — ทดลอง Capacity-Aware Gate

โน้ตบุ๊กนี้รันชุดตรวจสอบและการทดลอง **validation** จากแพ็กเกจ `Adaptive_Nervous_System_v020.zip` บน CPU ใช้ได้ใน Google Colab รวมถึงเปิดจากโทรศัพท์ ไม่ต้องมี GPU หรือ API key

**สิ่งที่จะเปรียบเทียบ 6 แบบ**

| ชื่อในผลลัพธ์ | ความหมาย |
| --- | --- |
| `static` | จัดสรรทรัพยากรคงที่ 69 / 69 / 69 |
| `maxweight` | เลือกการย้ายด้วยน้ำหนักงานค้างแบบง่าย |
| `abc` | ABC รุ่นเดิม |
| `abc_guard` | ABC พร้อม Gate ที่รอให้งานค้างต่อเนื่อง |
| `abc_capacity` | ตรวจความเหมาะสมด้านกำลังการทำงานเฉพาะข้อเสนอจาก ABC |
| `capacity_aware` | ใช้แบบจำลองกำลังการทำงานเลือกและตรวจการย้ายจากชุดการกระทำเดิม |

**ขอบเขต:** นี่คือการจำลองคิวและทรัพยากร ไม่ได้ควบคุมบริการจริง และไม่ใช่การพยากรณ์ประสิทธิภาพของระบบที่ใช้งานจริง ผลจากข้อมูลสังเคราะห์ต้องอ่านแยกตามกลุ่มสถานการณ์

**วิธีใช้:** เลือก CPU ในเมนู Runtime หากจำเป็น แล้วรันเซลล์ตามลำดับ เมื่อระบบถาม ให้อัปโหลดไฟล์ ZIP รุ่น v0.2.0 ที่ได้รับมาพร้อมโน้ตบุ๊กนี้ แต่ละครั้งจะสร้างโฟลเดอร์ผลลัพธ์ใหม่เพื่อรันการทดลองจริง ไม่ใช้ผลที่เก็บไว้จากครั้งก่อน

แพ็กเกจมี protocol และรายการตรวจสอบความถูกต้องที่ตรึงไว้แล้ว จึงไม่ต้องเรียกคำสั่ง freeze หรือแก้ค่าเพื่อให้ผ่าน validation ส่วนติดต่อของ Colab ยังไม่ได้ผ่านการทดสอบโดยตรงในสภาพแวดล้อมนี้

## 1. อัปโหลดและเปิดแพ็กเกจ

เลือกไฟล์ `Adaptive_Nervous_System_v020.zip` เพียงไฟล์เดียว โค้ดตรวจเส้นทางใน ZIP ก่อนแตกไฟล์ แล้วหาโครงการจาก `run_capacity.py` โดยอัตโนมัติ

In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import io
import stat
import tempfile
import zipfile

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("กรุณาเลือกไฟล์ Adaptive_Nervous_System_v020.zip เพียงไฟล์เดียว")
upload_name, upload_bytes = next(iter(uploaded.items()))
if not upload_name.lower().endswith(".zip"):
    raise ValueError("ไฟล์ที่เลือกต้องเป็น ZIP")

EXTRACT_ROOT = Path(tempfile.mkdtemp(prefix="ans_v020_")).resolve()
with zipfile.ZipFile(io.BytesIO(upload_bytes)) as archive:
    members = archive.infolist()
    if len(members) > 10000 or sum(m.file_size for m in members) > 512 * 1024 * 1024:
        raise ValueError("ขนาดแพ็กเกจเกินขอบเขตที่คาดไว้")
    destinations = set()
    for member in members:
        relative = PurePosixPath(member.filename)
        target = (EXTRACT_ROOT / member.filename).resolve()
        mode = member.external_attr >> 16
        if (relative.is_absolute() or ".." in relative.parts
                or "\\" in member.filename
                or not target.is_relative_to(EXTRACT_ROOT)
                or stat.S_ISLNK(mode)):
            raise ValueError(f"พบเส้นทางที่ไม่อนุญาตใน ZIP: {member.filename!r}")
        if target in destinations:
            raise ValueError(f"พบเส้นทางซ้ำใน ZIP: {member.filename!r}")
        destinations.add(target)
    archive.extractall(EXTRACT_ROOT)

candidates = sorted(EXTRACT_ROOT.rglob("run_capacity.py"))
if len(candidates) != 1:
    raise ValueError("ไม่พบโครงการ v0.2.0 เพียงหนึ่งชุด โปรดตรวจว่าเลือก ZIP รุ่นที่ถูกต้อง")
PROJECT = candidates[0].parent
for required in ("requirements.txt", "test_lab.py", "test_capacity.py"):
    if not (PROJECT / required).is_file():
        raise ValueError(f"แพ็กเกจไม่ครบ: ไม่พบ {required}")
print("พร้อมรันโครงการ:", PROJECT.name)

## 2. ติดตั้งส่วนที่จำเป็นและตรวจสอบโค้ด

ติดตั้งตาม `requirements.txt` ของแพ็กเกจ แล้วรันทั้งการตรวจต้นแบบเดิมและ Capacity Gate หากมีการตรวจใดไม่ผ่าน เซลล์จะหยุดพร้อมข้อความผิดพลาด ควรแก้สาเหตุก่อนใช้ผลทดลอง

In [ ]:
import subprocess
import sys

def run_python(*arguments):
    command = [sys.executable, *map(str, arguments)]
    print("กำลังรัน:", " ".join(command), flush=True)
    subprocess.run(command, cwd=PROJECT, check=True)

run_python("-m", "pip", "install", "-r", PROJECT / "requirements.txt")
run_python("-m", "unittest", "-v", "test_lab", "test_capacity")

## 3. รัน validation

การทดลองใช้ 11 กลุ่มสถานการณ์ กลุ่มละ 16 seed โดยเปรียบเทียบทั้ง 6 วิธีภายใต้งานเข้าและสภาพแวดล้อมเดียวกัน ABC แต่ละแบบใช้ 3 seed ก่อนเฉลี่ยภายในสถานการณ์ รวม **176 สถานการณ์ / 2,112 episode**

แต่ละครั้งที่รันเซลล์นี้จะสร้างโฟลเดอร์ใหม่ เพื่อไม่ให้การใช้ผลเดิมของ workflow ถูกนับเป็นการทดลองซ้ำ การรันอาจใช้เวลาหลายนาทีตามเครื่องของ Colab

In [ ]:
from datetime import datetime, timezone
import uuid

run_name = datetime.now(timezone.utc).strftime("validation_%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]
OUTPUT_ROOT = Path("/content/ans_capacity_runs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS = OUTPUT_ROOT / run_name
if RESULTS.exists():
    raise FileExistsError("ต้องใช้โฟลเดอร์ใหม่สำหรับการทดลอง")
run_python("run_capacity.py", "--split", "validation", "--out", RESULTS)
if not (RESULTS / "summary.json").is_file() or not (RESULTS / "episodes.csv").is_file():
    raise RuntimeError("ไม่พบไฟล์ผลลัพธ์ที่คาดไว้ กรุณาตรวจข้อความจากการรัน")
print("การทดลองเสร็จแล้ว:", run_name)

## 4. อ่านผลรวมและผลแต่ละแหล่งข้อมูล

ค่า **Mean backlog** คือจำนวนงานค้างเฉลี่ยเมื่อจบแต่ละนาที ยิ่งต่ำยิ่งดี ส่วน **Moves** คือจำนวนครั้งที่ย้ายทรัพยากร เฉลี่ยต่อ episode ผลรวมเป็นค่าเฉลี่ยจากครอบครัวการจำลองที่กำหนดไว้ จึงควรตรวจกลุ่มเดิมและกลุ่มที่สร้างขึ้นใหม่แยกกันด้วย

ความแตกต่างของ `abc_capacity` กับ `capacity_aware` ช่วยแยกผลของการกรองข้อเสนอ ABC ออกจากผลของการเปลี่ยนวิธีเลือกข้อเสนอ

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

summary = json.loads((RESULTS / "summary.json").read_text(encoding="utf-8"))
policy_order = ["static", "maxweight", "abc", "abc_guard", "abc_capacity", "capacity_aware"]
policy_names = {
    "static": "Static",
    "maxweight": "Queue-weight heuristic",
    "abc": "Original ABC",
    "abc_guard": "ABC + persistent gate",
    "abc_capacity": "ABC + capacity gate",
    "capacity_aware": "Capacity-aware selection",
}
metric_names = {
    "mean_backlog": "Mean backlog",
    "peak_backlog": "Mean peak backlog",
    "end_backlog": "Mean final backlog",
    "completed": "Mean completed work",
    "moves": "Mean moves",
}
overall = pd.DataFrame.from_dict(summary["overall"], orient="index").reindex(policy_order)
display(overall[list(metric_names)].rename(index=policy_names, columns=metric_names).round(3))
print("Scenarios:", summary.get("scenarios"), "| Episodes:", summary.get("episodes"),
      "| Simulated minutes:", summary.get("minutes"))

colors = ["#8293A6", "#8293A6", "#8293A6", "#8293A6", "#4D7EB5", "#15856B"]
fig, axes = plt.subplots(2, 1, figsize=(10, 8), constrained_layout=True)
for axis, metric, title in [
    (axes[0], "mean_backlog", "Mean end-of-minute backlog (lower is better)"),
    (axes[1], "moves", "Mean resource transfers per episode"),
]:
    axis.barh([policy_names[p] for p in policy_order], overall[metric], color=colors)
    axis.invert_yaxis()
    axis.set_title(title)
    axis.grid(axis="x", alpha=0.2)
    axis.set_axisbelow(True)
plt.show()

for source, values in summary.get("source_slices", {}).items():
    print("\nSource:", source, "| Scenarios:", values.get("scenarios"))
    table = pd.DataFrame.from_dict(values["overall"], orient="index").reindex(policy_order)
    display(table[list(metric_names)].rename(index=policy_names, columns=metric_names).round(3))

## 5. ตรวจความแตกต่างเทียบกับวิธีอื่น

เปอร์เซ็นต์ที่เป็นบวกหมายถึง Capacity-aware selection มีงานค้างเฉลี่ยน้อยกว่าวิธีที่เปรียบเทียบ ช่วงความเชื่อมั่น 95% มาจากการสุ่มซ้ำสถานการณ์ภายในแต่ละกลุ่ม หลังเฉลี่ย seed ของ ABC แล้ว ไม่ใช่ช่วงรับประกันผลกับระบบจริง

ค่าเปอร์เซ็นต์อาจไม่มีค่าเมื่อวิธีฐานมีงานค้างเป็นศูนย์ และผลรวมที่ดีกว่าไม่ได้แปลว่าดีกว่าในทุกกลุ่ม

In [ ]:
comparisons = summary.get("capacity_aware_vs", summary.get("comparisons", {}).get("capacity_aware", {}))
comparison_rows = []
for baseline, result in comparisons.items():
    interval = result.get("reduction_pct_ci95")
    comparison_rows.append({
        "Baseline": policy_names.get(baseline, baseline),
        "Backlog reduction (%)": result.get("reduction_pct"),
        "95% CI low": interval[0] if interval else None,
        "95% CI high": interval[1] if interval else None,
        "Paired wins": result.get("wins"),
        "Ties": result.get("ties"),
        "Losses": result.get("losses"),
    })
display(pd.DataFrame(comparison_rows).round(3))

group_rows = []
for group, values in summary.get("by_group", {}).items():
    group_overall = values.get("overall", {})
    group_rows.append({
        "Scenario family": group,
        **{policy_names[p]: group_overall.get(p, {}).get("mean_backlog") for p in policy_order},
    })
display(pd.DataFrame(group_rows).round(3))
print("รายละเอียดวิธีคำนวณ:", summary.get("analysis", {}).get("interval_scope", "ดูรายงานในแพ็กเกจ"))

## 6. ดาวน์โหลดผลที่รันใหม่

ไฟล์ ZIP นี้รวม `summary.json`, `episodes.csv` และหลักฐานการรันที่โปรแกรมสร้างไว้ เก็บคู่กับ ZIP ต้นฉบับและโน้ตบุ๊กเพื่อใช้ตรวจสอบหรือทดลองซ้ำได้

In [ ]:
import shutil

results_archive = Path(shutil.make_archive(
    str(OUTPUT_ROOT / (run_name + "_results")),
    "zip",
    root_dir=RESULTS,
))
print("ดาวน์โหลดผลการทดลอง:", results_archive.name)
files.download(str(results_archive))